# ATLAS Wind Atlas: delta scenarios

This notebook applies the climate change delta method to downscaled wind components.

It combines three inputs:

1. historical downscaled CMIP6 wind component
2. future downscaled CMIP6 wind component
3. present day integrated reanalysis wind component

The output is a future corrected wind component for each month and scenario.

The same workflow can be used for both `u` and `v` wind components. In the ATLAS wind pipeline, this notebook should be run after scenario downscaling and before the wind conversion and plotting notebooks.

## Method overview

For each month and climate scenario, the notebook computes the projected change between the future and historical CMIP6 simulations:

`delta = future_downscaled_component minus historical_downscaled_component`

The delta is then added to the present day reanalysis based integrated product:

`future_corrected_component = present_integrated_component plus delta`

This gives a future wind component that keeps the high resolution present day spatial structure while adding the projected climate change signal from CMIP6.

In [1]:
from pathlib import Path
import gc
import warnings

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore")

## User settings

Edit only this cell for a standard run.

`COUNTRY` is used in file names and folder names.

`MODEL` is the CMIP6 model already processed by the scenario downscaling notebook.

`START_YEAR` and `END_YEAR` define the future period. Historical is fixed to 1985 to 2014.

`AREA_MODE` is used only to keep the workflow readable. The actual spatial domain is selected through `AREA_NAME`, which must match the name used in the previous notebooks.

`COMPONENTS` can contain only `u`, only `v`, or both.

In [2]:
COUNTRY = "chile"
MODEL = "CNRM-ESM2-1"

START_YEAR = 2020
END_YEAR   = 2050

AREA_MODE = "continental"       # Use "islands" or "continental"
AREA_NAME = "continental"       # Must match the area name used in the previous notebooks

COMPONENT = "u"       # Wind component to process
EXPERIMENT = "ssp370"
MONTHS = range(1, 2)
MONTH  = 1

HISTORICAL_START_YEAR = 1985
HISTORICAL_END_YEAR = 2014

## Folder structure

The notebook uses relative paths so that it can be shared with other users without exposing local folders.

Expected input folders:

`../data/atlas_data/<country>/downscaling/historical/<model>/sub_areas/`

`../data/atlas_data/<country>/downscaling/<scenario>/<model>/sub_areas/`

`../data/atlas_data/<country>/scaling/sub_areas/`

Output folder:

`../data/atlas_data/<country>/scaling/<scenario>/<model>/sub_areas/`

In [3]:
PROJECT_DIR = Path("../data/atlas_data") / COUNTRY
SUBAREA_FOLDER = "sub_areas"
REAN_DIR    = PROJECT_DIR / SUBAREA_FOLDER 
OUTPUT_ROOT = PROJECT_DIR / MODEL / EXPERIMENT / SUBAREA_FOLDER

DOWNSCALING_DIR = Path("../data/downscaled_data") / COUNTRY / MODEL
print(f"Project folder: {PROJECT_DIR}")
print(f"Area mode: {AREA_MODE}")
print(f"Area name: {AREA_NAME}")
print(f"Model: {MODEL}")
print(f"Future period: {START_YEAR} to {END_YEAR}")

Project folder: ../data/atlas_data/chile
Area mode: continental
Area name: continental
Model: CNRM-ESM2-1
Future period: 2020 to 2050


## Helper functions

These functions keep file opening, coordinate checks and saving consistent across all months, scenarios and wind components.

In [4]:
def save_xarray_netcdf_fast(ds, output_path):
    """Save an xarray Dataset as compressed NetCDF using conservative chunk sizes."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    ds = ds.astype("float32")

    encoding = {}
    for var in ds.data_vars:
        dims = ds[var].dims
        shape = ds[var].shape
        chunksizes = []

        for dim, size in zip(dims, shape):
            if dim in ["latitude", "lat", "y"]:
                chunksizes.append(min(2048, size))
            elif dim in ["longitude", "lon", "x"]:
                chunksizes.append(min(2048, size))
            else:
                chunksizes.append(min(1, size))

        encoding[var] = {
            "dtype": "float32",
            "zlib": True,
            "complevel": 1,
            "shuffle": True,
            "chunksizes": tuple(chunksizes),
        }

    delayed = ds.to_netcdf(
        output_path,
        engine="h5netcdf",
        encoding=encoding,
        compute=False,
        mode="w",
    )

    with ProgressBar():
        delayed.compute(scheduler="single-threaded")


def get_spatial_coordinate_names(ds):
    """Return the latitude and longitude coordinate names used by a dataset."""
    lat_candidates = ["latitude", "lat", "y"]
    lon_candidates = ["longitude", "lon", "x"]

    lat_name = next((name for name in lat_candidates if name in ds.coords or name in ds.dims), None)
    lon_name = next((name for name in lon_candidates if name in ds.coords or name in ds.dims), None)

    if lat_name is None or lon_name is None:
        raise ValueError(
            "Could not identify latitude and longitude coordinates. "
            f"Available coordinates are: {list(ds.coords)}"
        )

    return lat_name, lon_name


def sort_spatial_coordinates(ds):
    """Sort the dataset by latitude and longitude when these coordinates are present."""
    lat_name, lon_name = get_spatial_coordinate_names(ds)

    if lat_name in ds.coords:
        ds = ds.sortby(lat_name)
    if lon_name in ds.coords:
        ds = ds.sortby(lon_name)

    return ds


def check_required_file(path):
    """Stop the workflow with a clear message if an expected input file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            "Required input file not found:"f"{path}"
            "Check COUNTRY, MODEL, AREA_NAME, START_YEAR, END_YEAR and the folder structure."
        )
    return path

## Input and output file names

These functions build file names from the user settings. If the previous notebooks use a different naming convention, update only these functions.

In [5]:
def cmip_downscaled_file(experiment, component, month):
    """Return the path to a downscaled CMIP6 component file."""
    if experiment == "historical":
        start_year = HISTORICAL_START_YEAR
        end_year = HISTORICAL_END_YEAR
    else:
        start_year = START_YEAR
        end_year = END_YEAR

    file_name = (
        f"{component}as_component_downscaled_"
        f"{COUNTRY}_m{month}_{AREA_NAME}_{start_year}_{end_year}.nc"
    )

    return DOWNSCALING_DIR / experiment / file_name


def present_integrated_file(component, month):
    """Return the path to the present day integrated reanalysis component file."""
    file_name = f"{component}10_integrated_{COUNTRY}_m{month}_{AREA_NAME}.nc"
    return REAN_DIR / file_name


def future_corrected_file(experiment, component, month):
    """Return the output path for the corrected future wind component."""
    file_name = (
        f"{component}10_corrected_"
        f"{COUNTRY}_m{month}_{experiment}_{AREA_NAME}_{START_YEAR}_{END_YEAR}.nc"
    )

    return OUTPUT_ROOT / file_name

## Delta calculation

The function below opens the three required files, aligns them on their spatial grid and saves the corrected future component.

In [6]:
def open_dataset_checked(path, chunks=None):
    """Open a NetCDF file after checking that it exists."""
    path = check_required_file(path)
    return xr.open_dataset(path, chunks=chunks)


def get_first_available_variable(ds, candidates):
    """Return the first variable name that exists in a dataset."""
    for candidate in candidates:
        if candidate in ds.data_vars:
            return candidate

    raise KeyError(
        "None of the expected variables were found."
        f"Expected one of: {candidates}"
        f"Available variables are: {list(ds.data_vars)}"
    )


def calculate_future_change(experiment, component, month):
    """Calculate and save the future corrected wind component for one month."""
    print(f"Processing {component} component, {experiment}, month {month:02d}")

    hist_path = cmip_downscaled_file("historical", component, month)
    future_path = cmip_downscaled_file(experiment, component, month)
    present_path = present_integrated_file(component, month)
    output_path = future_corrected_file(experiment, component, month)

    hist_ds = open_dataset_checked(hist_path, chunks={"latitude": 1000, "longitude": 1000})
    future_ds = open_dataset_checked(future_path, chunks={"latitude": 1000, "longitude": 1000})
    present_ds = open_dataset_checked(present_path, chunks={"latitude": 1000, "longitude": 1000})

    hist_ds = sort_spatial_coordinates(hist_ds)
    future_ds = sort_spatial_coordinates(future_ds)
    present_ds = sort_spatial_coordinates(present_ds)

    downscaled_var = get_first_available_variable(
        hist_ds,
        [f"{component}as_downscaled", f"{component}as", f"{component}10_downscaled"],
    )

    future_downscaled_var = get_first_available_variable(
        future_ds,
        [f"{component}as_downscaled", f"{component}as", f"{component}10_downscaled"],
    )

    present_var = get_first_available_variable(
        present_ds,
        [f"{component}10_integrated", f"{component}10", f"{component}10_corrected"],
    )

    hist_component, future_component = xr.align(
        hist_ds[downscaled_var],
        future_ds[future_downscaled_var],
        join="inner",
    )

    delta = future_component - hist_component
    delta = delta.to_dataset(name="delta")

    present_component, delta_component = xr.align(
        present_ds[present_var],
        delta["delta"],
        join="inner",
    )

    corrected = present_component + delta_component
    output_ds = corrected.to_dataset(name=f"{component}10_corrected")
    output_ds["delta"] = delta_component

    output_ds[f"{component}10_corrected"].attrs.update(
        {
            "long_name": f"Future corrected 10 m wind {component} component",
            "method": "delta change added to present day integrated reanalysis",
            "experiment": experiment,
            "model": MODEL,
            "future_period": f"{START_YEAR} to {END_YEAR}",
            "historical_period": f"{HISTORICAL_START_YEAR} to {HISTORICAL_END_YEAR}",
        }
    )

    output_ds["delta"].attrs.update(
        {
            "long_name": f"Projected change in {component} wind component",
            "method": "future downscaled CMIP6 minus historical downscaled CMIP6",
        }
    )

    print(f"Saving: {output_path}")
    save_xarray_netcdf_fast(output_ds, output_path)

    hist_ds.close()
    future_ds.close()
    present_ds.close()

    del hist_ds, future_ds, present_ds, output_ds, delta
    gc.collect()

    return output_path

## Run the workflow

This cell processes all selected components, scenarios and months.

For a complete wind atlas scenario workflow, run both `u` and `v`. The conversion notebook will then use these corrected components to calculate wind speed and wind direction.

In [7]:
output_file = calculate_future_change(
    experiment=EXPERIMENT,
    component=COMPONENT,
    month=MONTH,
)

print("Done.")


Processing u component, ssp370, month 01


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Saving: ../data/atlas_data/chile/CNRM-ESM2-1/ssp370/sub_areas/u10_corrected_chile_m1_ssp370_continental_2020_2050.nc
[########################################] | 100% Completed | 551.89 s
Done.


## Check the outputs

Use this cell to list the files produced by the workflow.

In [8]:
output_file

PosixPath('../data/atlas_data/chile/CNRM-ESM2-1/ssp370/sub_areas/u10_corrected_chile_m1_ssp370_continental_2020_2050.nc')

## Next step

After this notebook, run the wind conversion notebook to calculate wind speed from the corrected `u` and `v` components.